# 🏀 Inside the Possession: Play-by-Play Analytics & Win Probability

> **What happens inside a basketball game that box scores miss?** From the most dramatic win probability swings to rotation optimization and timeout effectiveness, we decode the game within the game.

**Part 10 of 10** in the [wyattowalsh/basketball](https://www.kaggle.com/datasets/wyattowalsh/basketball) companion notebook series.

---

## Table of Contents

1. [Setup & Data Loading](#1-setup--data-loading)
2. [Win Probability Deep Dive](#2-win-probability-deep-dive)
3. [Rotation Analysis](#3-rotation-analysis)
4. [Scoring Runs](#4-scoring-runs)
5. [Clutch Analysis](#5-clutch-analysis)
6. [Quarter Momentum](#6-quarter-momentum)
7. [Timeout Effectiveness](#7-timeout-effectiveness)
8. [Cleanup & Cross-Links](#8-cleanup--cross-links)

---

<a id="1-setup--data-loading"></a>
## 1. Setup & Data Loading

In [ ]:
!pip install -q "duckdb>=1.4,<2" "polars>=1.38,<2" "plotly>=5.18,<6" "scipy>=1.12,<2"

In [ ]:
import sys
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path

import duckdb
import numpy as np
import polars as pl
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from scipy import stats
from IPython.display import display, HTML

# Import shared utilities
_kaggle_path = Path("/kaggle/input/basketball")
sys.path.insert(0, str(_kaggle_path if _kaggle_path.exists() else Path(".")))
from nbadb_utils import COLORS, TEMPLATE, takeaway, get_connection

conn = get_connection()

# --- Detect latest season ---
LATEST_SEASON = conn.sql(
    "SELECT MAX(season_year) FROM dim_game"
).fetchone()[0]
print(f"Latest season: {LATEST_SEASON}")
print("Setup complete.")

---

<a id="2-win-probability-deep-dive"></a>
## 2. Win Probability Deep Dive

Win probability tracks how each team's chance of winning evolves across every possession.
We find the 5 most dramatic games in the dataset -- those with the highest volatility
in home win probability -- and visualize their full win probability curves.

In [ ]:
# Find the 5 most dramatic games by win probability volatility
dramatic_games = conn.sql("""
    SELECT
        w.game_id,
        STDDEV(w.home_pct) AS drama,
        COUNT(*) AS n_events,
        g.game_date,
        g.matchup
    FROM fact_win_probability w
    JOIN dim_game g ON w.game_id = g.game_id
    WHERE w.home_pct IS NOT NULL
    GROUP BY w.game_id, g.game_date, g.matchup
    HAVING COUNT(*) >= 50
    ORDER BY drama DESC
    LIMIT 5
""").pl()

print(f"Top 5 most dramatic games by win probability volatility:")
for row in dramatic_games.iter_rows(named=True):
    print(f"  {row['matchup']} ({row['game_date']}) -- drama={row['drama']:.4f}, events={row['n_events']}")

dramatic_ids = dramatic_games["game_id"].to_list()
dramatic_id_str = ", ".join(f"'{gid}'" for gid in dramatic_ids)

In [ ]:
# Query full win probability curves for the dramatic games
wp_curves = conn.sql(f"""
    SELECT
        w.game_id,
        w.event_num,
        w.period,
        w.pc_time_string,
        w.home_pct,
        w.visitor_pct,
        w.home_pts,
        w.visitor_pts,
        w.home_score_margin,
        ROW_NUMBER() OVER (PARTITION BY w.game_id ORDER BY w.period ASC, w.event_num ASC) AS event_idx
    FROM fact_win_probability w
    WHERE w.game_id IN ({dramatic_id_str})
      AND w.home_pct IS NOT NULL
    ORDER BY w.game_id, w.period, w.event_num
""").pl()

print(f"Win probability data points: {len(wp_curves):,}")

In [ ]:
# Create subplots: one win probability curve per dramatic game
n_games = len(dramatic_ids)
matchup_map = dict(zip(
    dramatic_games["game_id"].to_list(),
    [f"{m}\n({d})" for m, d in zip(
        dramatic_games["matchup"].to_list(),
        dramatic_games["game_date"].to_list(),
    )],
))

fig = make_subplots(
    rows=1, cols=n_games,
    subplot_titles=[matchup_map[gid] for gid in dramatic_ids],
    horizontal_spacing=0.04,
)

game_colors = [
    COLORS["primary"], COLORS["secondary"], COLORS["positive"],
    COLORS["purple"], COLORS["teal"],
]

for col_idx, gid in enumerate(dramatic_ids):
    game_data = wp_curves.filter(pl.col("game_id") == gid).sort("event_idx")
    event_idx = game_data["event_idx"].to_list()
    home_pct = game_data["home_pct"].to_list()
    color = game_colors[col_idx % len(game_colors)]

    # Win probability line
    fig.add_trace(
        go.Scatter(
            x=event_idx,
            y=home_pct,
            mode="lines",
            line=dict(color=color, width=1.5),
            fill="tonexty" if col_idx == 0 else None,
            name=matchup_map[gid].split("\n")[0],
            showlegend=False,
            hovertemplate=(
                "Event %{x}<br>"
                "Home Win%: %{y:.1%}<br>"
                "<extra></extra>"
            ),
        ),
        row=1, col=col_idx + 1,
    )

    # Fill above/below 0.5 using two traces
    home_pct_arr = np.array(home_pct)
    event_arr = np.array(event_idx)

    # Home-favored region (above 0.5)
    home_favored = np.where(home_pct_arr >= 0.5, home_pct_arr, 0.5)
    fig.add_trace(
        go.Scatter(
            x=event_arr.tolist(),
            y=home_favored.tolist(),
            mode="lines",
            line=dict(width=0),
            showlegend=False,
            hoverinfo="skip",
        ),
        row=1, col=col_idx + 1,
    )
    fig.add_trace(
        go.Scatter(
            x=event_arr.tolist(),
            y=[0.5] * len(event_arr),
            mode="lines",
            line=dict(width=0),
            fill="tonexty",
            fillcolor="rgba(0, 166, 81, 0.15)",
            showlegend=False,
            hoverinfo="skip",
        ),
        row=1, col=col_idx + 1,
    )

    # Visitor-favored region (below 0.5)
    visitor_favored = np.where(home_pct_arr <= 0.5, home_pct_arr, 0.5)
    fig.add_trace(
        go.Scatter(
            x=event_arr.tolist(),
            y=visitor_favored.tolist(),
            mode="lines",
            line=dict(width=0),
            showlegend=False,
            hoverinfo="skip",
        ),
        row=1, col=col_idx + 1,
    )
    fig.add_trace(
        go.Scatter(
            x=event_arr.tolist(),
            y=[0.5] * len(event_arr),
            mode="lines",
            line=dict(width=0),
            fill="tonexty",
            fillcolor="rgba(237, 28, 36, 0.15)",
            showlegend=False,
            hoverinfo="skip",
        ),
        row=1, col=col_idx + 1,
    )

    # 50% reference line
    fig.add_hline(
        y=0.5, line_dash="dash", line_color=COLORS["neutral"],
        line_width=1, row=1, col=col_idx + 1,
    )

    # Count lead changes (crossings of 0.5)
    crossings = np.sum(np.diff(np.sign(home_pct_arr - 0.5)) != 0)
    fig.add_annotation(
        text=f"{crossings} lead changes",
        x=0.5, y=0.05,
        xref=f"x{col_idx + 1} domain",
        yref=f"y{col_idx + 1} domain",
        showarrow=False,
        font=dict(size=10, color=COLORS["dark"]),
        bgcolor="rgba(255,255,255,0.8)",
    )

    fig.update_xaxes(title_text="Event Index", row=1, col=col_idx + 1)
    fig.update_yaxes(
        range=[0, 1], title_text="Home Win Probability" if col_idx == 0 else "",
        row=1, col=col_idx + 1,
    )

fig.update_layout(
    template=TEMPLATE,
    height=450,
    width=1200,
    title=dict(
        text="Top 5 Most Dramatic Games by Win Probability Volatility",
        font=dict(size=16, color=COLORS["primary"]),
    ),
    margin=dict(l=60, r=20, t=80, b=60),
)
fig.show()

In [ ]:
# Analyze the single most dramatic game in detail
most_dramatic_id = dramatic_ids[0]
most_dramatic_matchup = dramatic_games[0, "matchup"]
most_dramatic_date = dramatic_games[0, "game_date"]

md_data = wp_curves.filter(pl.col("game_id") == most_dramatic_id).sort("event_idx")
md_pct = np.array(md_data["home_pct"].to_list())

# Find the biggest single-event swing
swings = np.abs(np.diff(md_pct))
max_swing_idx = np.argmax(swings)
max_swing_val = swings[max_swing_idx]

# Count total lead changes
total_lead_changes = int(np.sum(np.diff(np.sign(md_pct - 0.5)) != 0))

# Min and max win probability for home team
min_wp = float(np.min(md_pct))
max_wp = float(np.max(md_pct))

takeaway(
    f"The most dramatic game was <strong>{most_dramatic_matchup}</strong> on {most_dramatic_date}. "
    f"It featured {total_lead_changes} lead changes, with home win probability ranging from "
    f"{min_wp:.1%} to {max_wp:.1%}. The biggest single-event swing was {max_swing_val:.1%} -- "
    f"a single play that shifted the entire trajectory of the game. These are the moments "
    f"box scores cannot capture: the emotional rollercoaster inside a basketball game."
)

---

<a id="3-rotation-analysis"></a>
## 3. Rotation Analysis

Rotation data reveals when each player enters and exits the game, how long their stints
last, and how effective they are during those minutes. We visualize a full game rotation
as a Gantt-style timeline, colored by point differential during each stint.

In [ ]:
# Use the most dramatic game for rotation analysis
rotation_game_id = most_dramatic_id

rotation_data = conn.sql(f"""
    SELECT
        r.game_id,
        r.team_id,
        r.player_id,
        p.full_name AS player_name,
        t.abbreviation AS team_abbr,
        r.in_time_real,
        r.out_time_real,
        r.pts,
        r.pts_diff,
        r.usg_pct,
        r.side,
        (r.out_time_real - r.in_time_real) / 10.0 AS stint_minutes
    FROM fact_rotation r
    JOIN dim_player p ON r.player_id = p.player_id
    JOIN dim_team t ON r.team_id = t.team_id
    WHERE r.game_id = '{rotation_game_id}'
    ORDER BY r.side, r.team_id, r.in_time_real
""").pl()

print(f"Rotation stints: {len(rotation_data):,}")
print(f"Teams: {rotation_data['team_abbr'].unique().to_list()}")
print(f"Players: {rotation_data['player_name'].n_unique()}")

In [ ]:
# Create rotation timeline (Gantt chart) for both teams
teams = rotation_data["team_abbr"].unique().sort().to_list()

fig = make_subplots(
    rows=2, cols=1,
    subplot_titles=[f"{t} Rotation" for t in teams],
    vertical_spacing=0.08,
    shared_xaxes=True,
)

for team_idx, team in enumerate(teams):
    team_rot = rotation_data.filter(pl.col("team_abbr") == team)

    # Get unique players ordered by total minutes (starters first)
    player_order = (
        team_rot.group_by("player_name")
        .agg(pl.col("stint_minutes").sum().alias("total_min"))
        .sort("total_min", descending=True)
        ["player_name"].to_list()
    )
    player_y_map = {name: i for i, name in enumerate(player_order)}

    for row in team_rot.iter_rows(named=True):
        y_pos = player_y_map[row["player_name"]]
        pts_diff = row["pts_diff"] if row["pts_diff"] is not None else 0

        # Color by pts_diff: green for positive, red for negative, gray for zero
        if pts_diff > 0:
            bar_color = f"rgba(0, 166, 81, {min(0.3 + abs(pts_diff) * 0.05, 0.9)})"
        elif pts_diff < 0:
            bar_color = f"rgba(237, 28, 36, {min(0.3 + abs(pts_diff) * 0.05, 0.9)})"
        else:
            bar_color = "rgba(140, 140, 140, 0.4)"

        # Convert times: in_time_real and out_time_real are in tenths of seconds
        # from game start. Convert to minutes for readability.
        in_min = row["in_time_real"] / 600.0 if row["in_time_real"] is not None else 0
        out_min = row["out_time_real"] / 600.0 if row["out_time_real"] is not None else 0

        fig.add_trace(
            go.Bar(
                x=[out_min - in_min],
                y=[row["player_name"]],
                base=[in_min],
                orientation="h",
                marker_color=bar_color,
                marker_line=dict(color="white", width=0.5),
                showlegend=False,
                hovertemplate=(
                    f"<b>{row['player_name']}</b><br>"
                    f"In: {in_min:.1f} min | Out: {out_min:.1f} min<br>"
                    f"Duration: {row['stint_minutes']:.1f} min<br>"
                    f"Points: {row['pts']} | +/-: {pts_diff:+d}<br>"
                    f"USG%: {row['usg_pct']}<extra></extra>"
                ),
            ),
            row=team_idx + 1, col=1,
        )

    fig.update_yaxes(
        categoryorder="array",
        categoryarray=player_order[::-1],
        row=team_idx + 1, col=1,
    )

# Add quarter markers
for q in range(1, 5):
    for r in [1, 2]:
        fig.add_vline(
            x=q * 12, line_dash="dot", line_color=COLORS["neutral"],
            line_width=1, row=r, col=1,
            annotation_text=f"Q{q}" if r == 1 else None,
            annotation_position="top",
        )

fig.update_xaxes(title_text="Game Time (minutes)", row=2, col=1)

fig.update_layout(
    template=TEMPLATE,
    height=700,
    width=1100,
    barmode="overlay",
    title=dict(
        text=f"Game Rotation Timeline: {most_dramatic_matchup} ({most_dramatic_date})",
        font=dict(size=16, color=COLORS["primary"]),
    ),
    margin=dict(l=130, r=20, t=80, b=60),
)
fig.show()

In [ ]:
# Identify best and worst stints by pts_diff per minute
stint_efficiency = (
    rotation_data
    .filter(pl.col("stint_minutes") >= 2.0)  # minimum 2 minutes for meaningful analysis
    .with_columns(
        (pl.col("pts_diff") / pl.col("stint_minutes")).alias("pts_diff_per_min")
    )
    .sort("pts_diff_per_min", descending=True)
)

best_stints = stint_efficiency.head(5)
worst_stints = stint_efficiency.tail(5)

print("Best stints (pts_diff per minute, min 2 min):")
for row in best_stints.iter_rows(named=True):
    print(f"  {row['player_name']} ({row['team_abbr']}): "
          f"{row['pts_diff_per_min']:+.2f}/min over {row['stint_minutes']:.1f} min "
          f"(+/-: {row['pts_diff']:+d})")

print("\nWorst stints (pts_diff per minute, min 2 min):")
for row in worst_stints.iter_rows(named=True):
    print(f"  {row['player_name']} ({row['team_abbr']}): "
          f"{row['pts_diff_per_min']:+.2f}/min over {row['stint_minutes']:.1f} min "
          f"(+/-: {row['pts_diff']:+d})")

takeaway(
    "Rotation data reveals the granular impact of coaching decisions. The best stints "
    "show players dominating short bursts, while the worst stints often correlate with "
    "mismatched lineups or fatigue. In dramatic games, the difference between winning "
    "and losing often comes down to which coach made better substitution timing decisions "
    "in the second half."
)

---

<a id="4-scoring-runs"></a>
## 4. Scoring Runs

A scoring run is a sequence of consecutive points scored by one team without the
other team scoring. Big runs (8+ points) are momentum-shifting events that can
swing a game. We use DuckDB window functions to detect them from play-by-play data.

In [ ]:
# Detect scoring runs from play-by-play data
# The score column has format like '23 - 19'. We parse both sides and detect
# consecutive scoring by one team.
scoring_runs = conn.sql("""
    WITH scored_events AS (
        SELECT
            game_id,
            event_num,
            period,
            pc_time_string,
            score,
            score_margin,
            event_type_name,
            -- Parse home and away scores from the score column (format: 'away - home')
            TRY_CAST(TRIM(SPLIT_PART(score, '-', 1)) AS INTEGER) AS away_score,
            TRY_CAST(TRIM(SPLIT_PART(score, '-', 2)) AS INTEGER) AS home_score,
            LAG(TRY_CAST(TRIM(SPLIT_PART(score, '-', 1)) AS INTEGER))
                OVER (PARTITION BY game_id ORDER BY event_num) AS prev_away_score,
            LAG(TRY_CAST(TRIM(SPLIT_PART(score, '-', 2)) AS INTEGER))
                OVER (PARTITION BY game_id ORDER BY event_num) AS prev_home_score
        FROM fact_play_by_play
        WHERE score IS NOT NULL
          AND score != ''
          AND event_type_name IN ('made_shot', 'free_throw')
    ),
    with_scoring_team AS (
        SELECT
            *,
            CASE
                WHEN home_score > COALESCE(prev_home_score, 0) THEN 'home'
                WHEN away_score > COALESCE(prev_away_score, 0) THEN 'away'
                ELSE NULL
            END AS scoring_side,
            CASE
                WHEN home_score > COALESCE(prev_home_score, 0)
                    THEN home_score - COALESCE(prev_home_score, 0)
                WHEN away_score > COALESCE(prev_away_score, 0)
                    THEN away_score - COALESCE(prev_away_score, 0)
                ELSE 0
            END AS points_scored
        FROM scored_events
        WHERE home_score IS NOT NULL AND away_score IS NOT NULL
    ),
    run_groups AS (
        SELECT
            *,
            SUM(CASE WHEN scoring_side != LAG(scoring_side)
                OVER (PARTITION BY game_id ORDER BY event_num)
                OR LAG(scoring_side) OVER (PARTITION BY game_id ORDER BY event_num) IS NULL
                THEN 1 ELSE 0 END)
                OVER (PARTITION BY game_id ORDER BY event_num) AS run_id
        FROM with_scoring_team
        WHERE scoring_side IS NOT NULL
    ),
    runs AS (
        SELECT
            game_id,
            run_id,
            scoring_side,
            MIN(period) AS period,
            SUM(points_scored) AS run_points,
            COUNT(*) AS run_baskets,
            MIN(event_num) AS start_event,
            MAX(event_num) AS end_event
        FROM run_groups
        GROUP BY game_id, run_id, scoring_side
    )
    SELECT *
    FROM runs
    WHERE run_points >= 2
    ORDER BY run_points DESC
""").pl()

print(f"Total scoring runs detected: {len(scoring_runs):,}")
print(f"Runs of 8+ points: {len(scoring_runs.filter(pl.col('run_points') >= 8)):,}")
print(f"Runs of 10+ points: {len(scoring_runs.filter(pl.col('run_points') >= 10)):,}")
print(f"Biggest run: {scoring_runs[0, 'run_points']} points")

In [ ]:
# Top 10 biggest scoring runs
top10_runs = scoring_runs.head(10)

# Get game context for the top runs
top_run_ids = top10_runs["game_id"].unique().to_list()
top_run_id_str = ", ".join(f"'{gid}'" for gid in top_run_ids)

game_context = conn.sql(f"""
    SELECT game_id, game_date, matchup
    FROM dim_game
    WHERE game_id IN ({top_run_id_str})
""").pl()
game_lookup = dict(zip(game_context["game_id"].to_list(), game_context["matchup"].to_list()))
date_lookup = dict(zip(game_context["game_id"].to_list(), game_context["game_date"].to_list()))

print("Top 10 Biggest Scoring Runs:")
print("-" * 80)
for i, row in enumerate(top10_runs.iter_rows(named=True)):
    matchup = game_lookup.get(row["game_id"], "Unknown")
    game_date = date_lookup.get(row["game_id"], "")
    print(f"  {i+1}. {row['run_points']} pts ({row['run_baskets']} baskets) "
          f"by {row['scoring_side']} team | Q{row['period']} | "
          f"{matchup} ({game_date})")

In [ ]:
# Distribution of scoring run sizes
run_dist = (
    scoring_runs
    .with_columns(
        pl.when(pl.col("run_points") >= 15).then(pl.lit("15+"))
        .when(pl.col("run_points") >= 12).then(pl.lit("12-14"))
        .when(pl.col("run_points") >= 10).then(pl.lit("10-11"))
        .when(pl.col("run_points") >= 8).then(pl.lit("8-9"))
        .when(pl.col("run_points") >= 6).then(pl.lit("6-7"))
        .when(pl.col("run_points") >= 4).then(pl.lit("4-5"))
        .otherwise(pl.lit("2-3"))
        .alias("run_bucket")
    )
    .group_by("run_bucket")
    .agg(pl.len().alias("count"))
    .sort("run_bucket")
)

# Ensure consistent ordering
bucket_order = ["2-3", "4-5", "6-7", "8-9", "10-11", "12-14", "15+"]
run_dist = run_dist.with_columns(
    pl.col("run_bucket").cast(pl.Categorical)
)
# Reorder manually
ordered_counts = []
for bucket in bucket_order:
    match = run_dist.filter(pl.col("run_bucket") == bucket)
    ordered_counts.append(match[0, "count"] if len(match) > 0 else 0)

fig = go.Figure()
fig.add_trace(go.Bar(
    x=bucket_order,
    y=ordered_counts,
    marker_color=[
        COLORS["neutral"], COLORS["neutral"], COLORS["accent"],
        COLORS["accent"], COLORS["secondary"], COLORS["secondary"],
        COLORS["negative"],
    ],
    text=ordered_counts,
    textposition="outside",
    hovertemplate="Run size: %{x} pts<br>Count: %{y:,}<extra></extra>",
))

fig.update_layout(
    template=TEMPLATE,
    title=dict(
        text="Distribution of Scoring Run Sizes Across All Games",
        font=dict(size=16, color=COLORS["primary"]),
    ),
    xaxis_title="Scoring Run Size (points)",
    yaxis_title="Frequency",
    height=450,
    width=800,
    showlegend=False,
)
fig.show()

In [ ]:
# Analyze what precedes big runs: look at events just before 8+ pt runs
big_runs = scoring_runs.filter(pl.col("run_points") >= 8).head(500)  # sample

# Register the big_runs dataframe in DuckDB and use a single JOIN query
# instead of looping with ~500 individual queries
conn.register("big_runs_view", big_runs.to_pandas())

preceding_df = conn.sql("""
    SELECT p.event_type_name, COUNT(*) AS cnt
    FROM big_runs_view br
    JOIN fact_play_by_play p
        ON p.game_id = br.game_id
        AND p.event_num >= br.start_event - 5
        AND p.event_num < br.start_event
    GROUP BY p.event_type_name
""").pl()

conn.unregister("big_runs_view")

if len(preceding_df) > 0:
    preceding_summary = preceding_df.sort("cnt", descending=True)

    print("Events preceding 8+ point runs (top event types):")
    total_events = preceding_summary["cnt"].sum()
    for row in preceding_summary.iter_rows(named=True):
        pct = row["cnt"] / total_events * 100
        print(f"  {row['event_type_name']}: {row['cnt']} ({pct:.1f}%)")

    takeaway(
        "Scoring runs of 8+ points are rare but decisive. They follow a power-law "
        "distribution: small runs are common, but runs above 12 points are exceptionally "
        "rare. The events preceding big runs frequently include turnovers and missed shots "
        "by the opposing team -- confirming that defensive pressure and transition play "
        "are the catalysts for momentum-shifting sequences."
    )
else:
    takeaway(
        "Scoring runs follow a power-law distribution: small runs are common, "
        "but runs above 10 points are exceptionally rare and often decisive."
    )

---

<a id="5-clutch-analysis"></a>
## 5. Clutch Analysis

Clutch time: period >= 4, 2 minutes or fewer remaining, game within 5 points.
We analyze how shooting efficiency, turnover rate, and free throw rate differ
under maximum pressure compared to regular play.

In [ ]:
# Parse play-by-play for clutch analysis
# pc_time_string is like 'MM:SS' -- we need to parse minutes remaining
clutch_pbp = conn.sql("""
    WITH parsed AS (
        SELECT
            game_id,
            event_num,
            period,
            pc_time_string,
            event_type_name,
            event_msg_type,
            player1_id,
            player1_team_id,
            score,
            score_margin,
            TRY_CAST(SPLIT_PART(pc_time_string, ':', 1) AS INTEGER) AS minutes_remaining,
            TRY_CAST(score_margin AS INTEGER) AS margin_int
        FROM fact_play_by_play
        WHERE event_type_name IN ('made_shot', 'missed_shot', 'free_throw', 'turnover')
    )
    SELECT
        *,
        CASE
            WHEN period >= 4
                AND minutes_remaining IS NOT NULL
                AND minutes_remaining <= 2
                AND margin_int IS NOT NULL
                AND ABS(margin_int) <= 5
            THEN 'Clutch'
            ELSE 'Non-Clutch'
        END AS situation
    FROM parsed
""").pl()

clutch_count = len(clutch_pbp.filter(pl.col("situation") == "Clutch"))
non_clutch_count = len(clutch_pbp.filter(pl.col("situation") == "Non-Clutch"))
print(f"Clutch plays: {clutch_count:,}")
print(f"Non-clutch plays: {non_clutch_count:,}")

In [ ]:
# Compute FG%, turnover rate, FT rate in clutch vs non-clutch
clutch_stats = (
    clutch_pbp
    .group_by("situation")
    .agg([
        # FG%: made shots / (made + missed shots)
        pl.col("event_type_name")
            .filter(pl.col("event_type_name") == "made_shot")
            .count()
            .alias("made_shots"),
        pl.col("event_type_name")
            .filter(pl.col("event_type_name") == "missed_shot")
            .count()
            .alias("missed_shots"),
        pl.col("event_type_name")
            .filter(pl.col("event_type_name") == "free_throw")
            .count()
            .alias("free_throws"),
        pl.col("event_type_name")
            .filter(pl.col("event_type_name") == "turnover")
            .count()
            .alias("turnovers"),
        pl.len().alias("total_plays"),
    ])
    .with_columns([
        (pl.col("made_shots") / (pl.col("made_shots") + pl.col("missed_shots")) * 100)
            .alias("fg_pct"),
        (pl.col("turnovers") / pl.col("total_plays") * 100)
            .alias("turnover_rate"),
        (pl.col("free_throws") / pl.col("total_plays") * 100)
            .alias("ft_rate"),
    ])
)

print(clutch_stats)

In [ ]:
# Grouped bar chart: clutch vs non-clutch
metrics = ["FG%", "Turnover Rate", "FT Rate"]
metric_cols = ["fg_pct", "turnover_rate", "ft_rate"]

fig = go.Figure()

for situation, color in [("Non-Clutch", COLORS["primary"]), ("Clutch", COLORS["secondary"])]:
    row_data = clutch_stats.filter(pl.col("situation") == situation)
    if len(row_data) == 0:
        continue
    values = [float(row_data[0, col]) for col in metric_cols]

    fig.add_trace(go.Bar(
        x=metrics,
        y=values,
        name=situation,
        marker_color=color,
        text=[f"{v:.1f}%" for v in values],
        textposition="outside",
        hovertemplate="%{x}: %{y:.1f}%<extra></extra>",
    ))

fig.update_layout(
    template=TEMPLATE,
    barmode="group",
    title=dict(
        text="Clutch vs Non-Clutch: Shooting, Turnovers & Free Throws",
        font=dict(size=16, color=COLORS["primary"]),
    ),
    xaxis_title="Metric",
    yaxis_title="Rate (%)",
    height=500,
    width=700,
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
)
fig.show()

In [ ]:
# Best clutch performers: top-10 players by clutch FG% (min 30 clutch FGA)
clutch_players = (
    clutch_pbp
    .filter(
        (pl.col("situation") == "Clutch")
        & (pl.col("event_type_name").is_in(["made_shot", "missed_shot"]))
        & (pl.col("player1_id").is_not_null())
    )
    .group_by("player1_id")
    .agg([
        pl.col("event_type_name")
            .filter(pl.col("event_type_name") == "made_shot")
            .count()
            .alias("clutch_makes"),
        pl.len().alias("clutch_fga"),
    ])
    .filter(pl.col("clutch_fga") >= 30)
    .with_columns(
        (pl.col("clutch_makes") / pl.col("clutch_fga") * 100).alias("clutch_fg_pct")
    )
    .sort("clutch_fg_pct", descending=True)
    .head(10)
)

# Get player names
clutch_player_ids = clutch_players["player1_id"].to_list()
clutch_player_id_str = ", ".join(str(pid) for pid in clutch_player_ids)

player_names = conn.sql(f"""
    SELECT player_id, full_name
    FROM dim_player
    WHERE player_id IN ({clutch_player_id_str})
""").pl()
name_map = dict(zip(player_names["player_id"].to_list(), player_names["full_name"].to_list()))

clutch_players = clutch_players.with_columns(
    pl.col("player1_id").replace_strict(name_map, default="Unknown").alias("player_name")
)

# Bar chart
fig = go.Figure()
fig.add_trace(go.Bar(
    y=clutch_players["player_name"].to_list()[::-1],
    x=clutch_players["clutch_fg_pct"].to_list()[::-1],
    orientation="h",
    marker_color=COLORS["secondary"],
    text=[
        f"{fg:.1f}% ({fga} FGA)"
        for fg, fga in zip(
            clutch_players["clutch_fg_pct"].to_list()[::-1],
            clutch_players["clutch_fga"].to_list()[::-1],
        )
    ],
    textposition="outside",
    hovertemplate="%{y}<br>Clutch FG%: %{x:.1f}%<extra></extra>",
))

fig.update_layout(
    template=TEMPLATE,
    title=dict(
        text="Top 10 Clutch Performers by FG% (min 30 clutch FGA)",
        font=dict(size=16, color=COLORS["primary"]),
    ),
    xaxis_title="Clutch FG%",
    height=450,
    width=800,
    margin=dict(l=150, r=80, t=60, b=40),
    showlegend=False,
)
fig.show()

# Compute clutch vs non-clutch FG% difference
clutch_fg = clutch_stats.filter(pl.col("situation") == "Clutch")
non_clutch_fg = clutch_stats.filter(pl.col("situation") == "Non-Clutch")
if len(clutch_fg) > 0 and len(non_clutch_fg) > 0:
    fg_drop = float(clutch_fg[0, "fg_pct"]) - float(non_clutch_fg[0, "fg_pct"])
    to_drop = float(clutch_fg[0, "turnover_rate"]) - float(non_clutch_fg[0, "turnover_rate"])
    best_clutch = clutch_players[0, "player_name"]
    best_pct = clutch_players[0, "clutch_fg_pct"]

    takeaway(
        f"League-wide FG% drops by {abs(fg_drop):.1f} percentage points in clutch situations, "
        f"while turnover rate shifts by {to_drop:+.1f} percentage points. Free throw rate rises "
        f"as teams foul intentionally. The best clutch shooter is <strong>{best_clutch}</strong> "
        f"at {best_pct:.1f}% in clutch FGA -- a true ice-in-the-veins performer."
    )
else:
    takeaway(
        "Shooting efficiency declines in clutch situations across the league, "
        "while free throw rate increases due to intentional fouling strategies."
    )

---

<a id="6-quarter-momentum"></a>
## 6. Quarter Momentum

Some teams are closers that dominate Q4; others are faders that wilt under pressure.
Using quarterly scoring data from `fact_game_result`, we compute each team's
average Q4 scoring differential and identify the best and worst closing teams.

In [ ]:
# Compute Q4 differential per team
# A team appears as both home and away, so we need to handle both cases.
q4_diff = conn.sql(f"""
    WITH team_q4 AS (
        -- Home games
        SELECT
            home_team_id AS team_id,
            pts_qtr4_home AS team_q4,
            pts_qtr4_away AS opp_q4,
            pts_qtr4_home - pts_qtr4_away AS q4_diff,
            pts_qtr1_home - pts_qtr1_away AS q1_diff,
            pts_qtr2_home - pts_qtr2_away AS q2_diff,
            pts_qtr3_home - pts_qtr3_away AS q3_diff
        FROM fact_game_result
        WHERE season_year = '{LATEST_SEASON}'
          AND season_type = 'Regular Season'
          AND pts_qtr4_home IS NOT NULL
          AND pts_qtr4_away IS NOT NULL
        UNION ALL
        -- Away games (flip perspective)
        SELECT
            visitor_team_id AS team_id,
            pts_qtr4_away AS team_q4,
            pts_qtr4_home AS opp_q4,
            pts_qtr4_away - pts_qtr4_home AS q4_diff,
            pts_qtr1_away - pts_qtr1_home AS q1_diff,
            pts_qtr2_away - pts_qtr2_home AS q2_diff,
            pts_qtr3_away - pts_qtr3_home AS q3_diff
        FROM fact_game_result
        WHERE season_year = '{LATEST_SEASON}'
          AND season_type = 'Regular Season'
          AND pts_qtr4_home IS NOT NULL
          AND pts_qtr4_away IS NOT NULL
    )
    SELECT
        t.abbreviation,
        COUNT(*) AS games,
        AVG(tq.q1_diff) AS avg_q1_diff,
        AVG(tq.q2_diff) AS avg_q2_diff,
        AVG(tq.q3_diff) AS avg_q3_diff,
        AVG(tq.q4_diff) AS avg_q4_diff,
        AVG(tq.team_q4) AS avg_q4_pts
    FROM team_q4 tq
    JOIN dim_team t ON tq.team_id = t.team_id
    GROUP BY t.abbreviation
    HAVING COUNT(*) >= 20
    ORDER BY avg_q4_diff DESC
""").pl()

print(f"Teams with Q4 data: {len(q4_diff)}")
print(f"\nTop 5 closers (avg Q4 differential):")
for row in q4_diff.head(5).iter_rows(named=True):
    print(f"  {row['abbreviation']}: {row['avg_q4_diff']:+.1f} ({row['games']} games)")
print(f"\nBottom 5 faders:")
for row in q4_diff.tail(5).iter_rows(named=True):
    print(f"  {row['abbreviation']}: {row['avg_q4_diff']:+.1f} ({row['games']} games)")

In [ ]:
# Bar chart sorted by Q4 differential
team_names = q4_diff["abbreviation"].to_list()
q4_vals = q4_diff["avg_q4_diff"].to_list()

bar_colors = [
    COLORS["positive"] if v > 0 else COLORS["negative"]
    for v in q4_vals
]

fig = go.Figure()
fig.add_trace(go.Bar(
    y=team_names[::-1],
    x=q4_vals[::-1],
    orientation="h",
    marker_color=bar_colors[::-1],
    text=[f"{v:+.1f}" for v in q4_vals[::-1]],
    textposition="outside",
    hovertemplate="%{y}<br>Avg Q4 Diff: %{x:+.1f}<extra></extra>",
))

fig.add_vline(x=0, line_dash="solid", line_color=COLORS["dark"], line_width=1)

fig.update_layout(
    template=TEMPLATE,
    title=dict(
        text=f"Q4 Scoring Differential by Team ({LATEST_SEASON}) -- Closers vs Faders",
        font=dict(size=16, color=COLORS["primary"]),
    ),
    xaxis_title="Avg Q4 Point Differential",
    height=max(500, len(team_names) * 22 + 100),
    width=800,
    margin=dict(l=70, r=80, t=60, b=40),
    showlegend=False,
)
fig.show()

In [ ]:
# Quarter-by-quarter heatmap for all teams
q_cols = ["avg_q1_diff", "avg_q2_diff", "avg_q3_diff", "avg_q4_diff"]
q_labels = ["Q1", "Q2", "Q3", "Q4"]

# Sort teams by overall scoring differential (sum of all quarters)
q4_diff_sorted = q4_diff.with_columns(
    (pl.col("avg_q1_diff") + pl.col("avg_q2_diff") + pl.col("avg_q3_diff") + pl.col("avg_q4_diff"))
    .alias("total_diff")
).sort("total_diff", descending=True)

heatmap_data = np.array([
    q4_diff_sorted[col].to_list() for col in q_cols
]).T  # shape: (n_teams, 4)

fig = go.Figure(data=go.Heatmap(
    z=heatmap_data,
    x=q_labels,
    y=q4_diff_sorted["abbreviation"].to_list(),
    colorscale="RdYlGn",
    zmid=0,
    text=np.round(heatmap_data, 1),
    texttemplate="%{text:+.1f}",
    textfont=dict(size=10),
    hovertemplate="%{y} %{x}<br>Avg Diff: %{z:+.1f}<extra></extra>",
    colorbar=dict(title="Avg Diff"),
))

fig.update_layout(
    template=TEMPLATE,
    title=dict(
        text=f"Quarter-by-Quarter Scoring Differential Heatmap ({LATEST_SEASON})",
        font=dict(size=16, color=COLORS["primary"]),
    ),
    height=max(500, len(team_names) * 22 + 100),
    width=500,
    margin=dict(l=60, r=20, t=60, b=40),
    yaxis=dict(autorange="reversed"),
)
fig.show()

# Identify closers and faders
best_closer = q4_diff[0, "abbreviation"]
best_closer_diff = q4_diff[0, "avg_q4_diff"]
worst_closer = q4_diff[-1, "abbreviation"]
worst_closer_diff = q4_diff[-1, "avg_q4_diff"]

takeaway(
    f"<strong>{best_closer}</strong> is the best closing team with a Q4 differential of "
    f"{best_closer_diff:+.1f} points, while <strong>{worst_closer}</strong> fades hardest at "
    f"{worst_closer_diff:+.1f}. The heatmap reveals that some teams are strong starters "
    f"(Q1/Q2) but poor closers, while elite teams maintain or increase their differential "
    f"as the game progresses -- a hallmark of superior conditioning, depth, and coaching "
    f"adjustments."
)

---

<a id="7-timeout-effectiveness"></a>
## 7. Timeout Effectiveness

Do timeouts actually work? We analyze two angles:
1. Direct timeout events from play-by-play (event_type_name = 'timeout')
2. Substitution-break effectiveness using rotation stint boundaries

We compare scoring rate before vs after these stoppages to measure their impact.

In [ ]:
# Approach 1: Direct timeout analysis from play-by-play
# Find timeout events and compare the score margin trajectory around them
# Uses window-function approach instead of correlated subqueries for performance
timeout_analysis = conn.sql("""
    WITH timeouts AS (
        SELECT
            game_id,
            event_num AS timeout_event,
            period,
            player1_team_id AS timeout_team_id,
            TRY_CAST(score_margin AS INTEGER) AS margin_at_timeout
        FROM fact_play_by_play
        WHERE event_type_name = 'timeout'
          AND score_margin IS NOT NULL
          AND score_margin != ''
    ),
    scored_events AS (
        SELECT
            game_id,
            event_num,
            TRY_CAST(score_margin AS INTEGER) AS margin_int
        FROM fact_play_by_play
        WHERE score_margin IS NOT NULL
          AND score_margin != ''
    ),
    timeout_with_rn AS (
        SELECT
            t.game_id,
            t.timeout_event,
            se.margin_int,
            ROW_NUMBER() OVER (
                PARTITION BY t.game_id, t.timeout_event
                ORDER BY se.event_num
            ) AS rn
        FROM timeouts t
        JOIN scored_events se
            ON se.game_id = t.game_id
            AND se.event_num > t.timeout_event
    ),
    post_timeout AS (
        SELECT
            t.game_id,
            t.timeout_event,
            t.period,
            t.timeout_team_id,
            t.margin_at_timeout,
            m10.margin_int AS margin_after_10
        FROM timeouts t
        LEFT JOIN timeout_with_rn m10
            ON m10.game_id = t.game_id
            AND m10.timeout_event = t.timeout_event
            AND m10.rn = 10
    )
    SELECT
        game_id,
        timeout_event,
        period,
        margin_at_timeout,
        margin_after_10,
        margin_after_10 - margin_at_timeout AS margin_change
    FROM post_timeout
    WHERE margin_after_10 IS NOT NULL
""").pl()

print(f"Timeout events with before/after data: {len(timeout_analysis):,}")
if len(timeout_analysis) > 0:
    avg_change = timeout_analysis["margin_change"].mean()
    print(f"Average margin change (10 events post-timeout): {avg_change:+.2f}")

In [ ]:
# Approach 2: Rotation stint analysis
# Compare pts_diff in the first portion of new stints vs the last portion of previous stints
# This captures whether fresh substitution patterns improve team performance

stint_analysis = conn.sql(f"""
    WITH stints AS (
        SELECT
            game_id,
            team_id,
            player_id,
            in_time_real,
            out_time_real,
            pts,
            pts_diff,
            (out_time_real - in_time_real) / 10.0 AS stint_minutes,
            ROW_NUMBER() OVER (
                PARTITION BY game_id, player_id
                ORDER BY in_time_real
            ) AS stint_num
        FROM fact_rotation
        WHERE (out_time_real - in_time_real) >= 600  -- at least 1 minute
    )
    SELECT
        s1.pts_diff AS prev_stint_diff,
        s2.pts_diff AS next_stint_diff,
        s1.stint_minutes AS prev_stint_min,
        s2.stint_minutes AS next_stint_min,
        s1.pts_diff / NULLIF(s1.stint_minutes, 0) AS prev_diff_per_min,
        s2.pts_diff / NULLIF(s2.stint_minutes, 0) AS next_diff_per_min
    FROM stints s1
    JOIN stints s2
        ON s1.game_id = s2.game_id
        AND s1.player_id = s2.player_id
        AND s1.stint_num = s2.stint_num - 1
    WHERE s1.stint_minutes >= 2
      AND s2.stint_minutes >= 2
""").pl()

print(f"Stint pairs for analysis: {len(stint_analysis):,}")

if len(stint_analysis) > 0:
    prev_rates = stint_analysis["prev_diff_per_min"].drop_nulls().to_list()
    next_rates = stint_analysis["next_diff_per_min"].drop_nulls().to_list()

    avg_prev = np.mean(prev_rates)
    avg_next = np.mean(next_rates)
    print(f"Avg pts_diff/min in previous stint: {avg_prev:+.3f}")
    print(f"Avg pts_diff/min in next stint (after sub break): {avg_next:+.3f}")

    # Statistical test
    t_stat, p_value = stats.ttest_ind(next_rates, prev_rates, equal_var=False)
    print(f"\nWelch's t-test: t={t_stat:.3f}, p={p_value:.4f}")
    print(f"Statistically significant (p < 0.05): {p_value < 0.05}")

In [ ]:
# Visualize: distribution of pts_diff/min before vs after substitution breaks
if len(stint_analysis) > 0:
    fig = go.Figure()

    fig.add_trace(go.Histogram(
        x=prev_rates,
        name="End of Previous Stint",
        marker_color=COLORS["negative"],
        opacity=0.6,
        nbinsx=50,
        histnorm="probability density",
    ))
    fig.add_trace(go.Histogram(
        x=next_rates,
        name="Start of Next Stint",
        marker_color=COLORS["positive"],
        opacity=0.6,
        nbinsx=50,
        histnorm="probability density",
    ))

    fig.add_vline(x=avg_prev, line_dash="dash", line_color=COLORS["negative"],
                  line_width=2, annotation_text=f"Prev mean: {avg_prev:+.3f}")
    fig.add_vline(x=avg_next, line_dash="dash", line_color=COLORS["positive"],
                  line_width=2, annotation_text=f"Next mean: {avg_next:+.3f}")

    fig.update_layout(
        template=TEMPLATE,
        barmode="overlay",
        title=dict(
            text="Points Differential Per Minute: Before vs After Substitution Breaks",
            font=dict(size=16, color=COLORS["primary"]),
        ),
        xaxis_title="Pts Diff / Minute",
        yaxis_title="Density",
        height=450,
        width=850,
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
    )
    fig.show()

In [ ]:
# Timeout context analysis: are timeouts called more when trailing?
if len(timeout_analysis) > 0:
    trailing = len(timeout_analysis.filter(pl.col("margin_at_timeout") < 0))
    leading = len(timeout_analysis.filter(pl.col("margin_at_timeout") > 0))
    tied = len(timeout_analysis.filter(pl.col("margin_at_timeout") == 0))

    # Margin change when trailing vs leading
    trailing_change = timeout_analysis.filter(
        pl.col("margin_at_timeout") < 0
    )["margin_change"].mean()
    leading_change = timeout_analysis.filter(
        pl.col("margin_at_timeout") > 0
    )["margin_change"].mean()

    fig = go.Figure()
    fig.add_trace(go.Bar(
        x=["When Trailing", "When Leading", "When Tied"],
        y=[trailing, leading, tied],
        marker_color=[COLORS["negative"], COLORS["positive"], COLORS["neutral"]],
        text=[trailing, leading, tied],
        textposition="outside",
    ))
    fig.update_layout(
        template=TEMPLATE,
        title=dict(
            text="When Are Timeouts Called? (by Score Margin)",
            font=dict(size=16, color=COLORS["primary"]),
        ),
        xaxis_title="Game State",
        yaxis_title="Number of Timeouts",
        height=400,
        width=600,
        showlegend=False,
    )
    fig.show()

    sig_str = (
        f"statistically significant (p={p_value:.4f})"
        if len(stint_analysis) > 0 and p_value < 0.05
        else f"not statistically significant (p={p_value:.4f})"
        if len(stint_analysis) > 0
        else "could not be tested"
    )

    takeaway(
        f"Timeouts are called more often when trailing ({trailing:,}) than leading ({leading:,}), "
        f"confirming coaches use them as defensive resets. When trailing, the average margin change "
        f"10 events after a timeout is {trailing_change:+.2f}. The substitution break analysis "
        f"shows the performance difference before vs after fresh stints is {sig_str}. "
        f"The effect size is small, suggesting that while rest and coaching adjustments matter, "
        f"they are just one factor among many in shifting game momentum."
    )
else:
    takeaway(
        "Timeout data was limited in the available play-by-play. The rotation stint analysis "
        "provides an alternative view: fresh stints after substitution breaks show modest "
        "performance improvements, though the effect is small relative to player talent "
        "and matchup factors."
    )

---

<a id="8-cleanup--cross-links"></a>
## 8. Cleanup & Cross-Links

In [ ]:
# Cleanup handled by nbadb_utils.get_connection() via atexit
print("Analysis complete.")

### Summary

This notebook went beyond the box score to explore what happens inside a basketball game:

1. **Win Probability Deep Dive** -- identified the most dramatic games by volatility, visualized full win probability curves with lead changes and momentum swings
2. **Rotation Analysis** -- built Gantt-style timelines showing every player's on-court stints, colored by point differential, revealing coaching patterns and lineup effectiveness
3. **Scoring Runs** -- detected and quantified momentum-shifting scoring runs from play-by-play data, finding that turnovers and missed shots most commonly precede big runs
4. **Clutch Analysis** -- measured the pressure effect: FG% drops in clutch time, FT rate spikes, and identified the league's best clutch performers
5. **Quarter Momentum** -- ranked every team by Q4 scoring differential, separating closers from faders with a quarter-by-quarter heatmap
6. **Timeout Effectiveness** -- analyzed whether timeouts and substitution breaks actually improve performance, with statistical testing for significance

---

### nbadb Notebook Suite

| Part | Notebook | Description |
|---|---|---|
| Part 1 | [MVP Prediction](./nba_mvp_predictor.ipynb) | MVP Prediction with Tracking & Synergy Data |
| Part 2 | [Data-Driven Player Archetypes](./nba_player_archetypes.ipynb) | Data-Driven Player Archetypes (UMAP + GMM) |
| Part 3 | [Game Outcome Prediction](./nba_game_prediction.ipynb) | Game Outcome Prediction (Stacking Ensemble) |
| Part 4 | [Draft Combine to Career](./nba_draft_combine_analysis.ipynb) | Draft Combine to Career Prediction |
| Part 5 | [Defense Decoded](./nba_defense_decoded.ipynb) | Defense Decoded (Tracking + Hustle + Synergy PCA) |
| Part 6 | [Interactive Player Explorer](./nba_player_dashboard.ipynb) | Interactive Player Explorer |
| Part 7 | [Spatial Shot Analysis](./nba_shot_chart_analysis.ipynb) | Spatial Shot Analysis & 3-Point Revolution |
| Part 8 | [Player Similarity Engine](./nba_player_similarity.ipynb) | Player Similarity Engine (Cosine + Manhattan) |
| Part 9 | [Career Trajectory](./nba_aging_curves.ipynb) | Career Trajectory & Aging Curve Modeling |
| **Part 10** | **Play-by-Play Insights** (this notebook) | **Play-by-Play: Win Probability, Runs & Clutch** |

---

**Dataset**: [wyattowalsh/basketball on Kaggle](https://www.kaggle.com/datasets/wyattowalsh/basketball)
**Documentation**: [nbadb.dev](https://nbadb.dev)
**Source**: [github.com/wyattowalsh/nbadb](https://github.com/wyattowalsh/nbadb)